<a href="https://colab.research.google.com/github/TahaNawazkhan/Artificial-intelligence-/blob/main/lab3%20task%201.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from collections import deque


class Node:
    def __init__(self, name):
        self.name = name
        self.children = []

    def add(self, *kids):
        self.children.extend(kids)
        return self


# ---------------- Tree construction ----------------
def build_sample_tree():
    #               A
    #        /      |      \
    #       B       C       D
    #     / | \    / \      |
    #    E  F  G  H   I     J
    #   /\     |      |    /|\
    #  K  L    M      N   O P Q
    n = {c: Node(c) for c in "ABCDEFGHIJKLMNOPQ"}
    n["A"].add(n["B"], n["C"], n["D"])
    n["B"].add(n["E"], n["F"], n["G"])
    n["C"].add(n["H"], n["I"])
    n["D"].add(n["J"])
    n["E"].add(n["K"], n["L"])
    n["G"].add(n["M"])
    n["I"].add(n["N"])
    n["J"].add(n["O"], n["P"], n["Q"])
    return n["A"]


def build_complete_tree(branching, depth):
    counter = [0]

    def make(d):
        node = Node(str(counter[0]))
        counter[0] += 1
        if d < depth:
            for _ in range(branching):
                node.children.append(make(d + 1))
        return node
    return make(0)


def count_nodes_to_depth(root, limit):
    if limit < 0:
        return 0
    return 1 + sum(count_nodes_to_depth(c, limit - 1) for c in root.children)


# ---------------- BFS ----------------
def bfs(root, goal):
    visited = 0
    queue = deque([root])
    while queue:
        node = queue.popleft()
        visited += 1                       # node examined
        if node.name == goal:
            return True, visited
        queue.extend(node.children)
    return False, visited


# ---------------- IDDFS ----------------
def depth_limited_search(node, goal, limit, counter):
    counter[0] += 1                        # node examined
    if node.name == goal:
        return True
    if limit == 0:
        return False
    for child in node.children:
        if depth_limited_search(child, goal, limit - 1, counter):
            return True
    return False


def iddfs(root, goal, max_depth=50):
    total = 0
    per_iter = []
    for limit in range(max_depth + 1):
        counter = [0]
        found = depth_limited_search(root, goal, limit, counter)
        per_iter.append((limit, counter[0]))
        total += counter[0]                # sum across ALL iterations
        if found:
            return True, total, per_iter
        # goal absent and whole tree already covered -> stop
        if count_nodes_to_depth(root, limit) == count_nodes_to_depth(root, limit + 1):
            break
    return False, total, per_iter


# ---------------- Comparison ----------------
def compare(root, goal, label=""):
    b_found, b_count = bfs(root, goal)
    i_found, i_count, per_iter = iddfs(root, goal)

    print(f"\nGoal = {goal!r} {label}")
    print("  IDDFS iterations:")
    for limit, c in per_iter:
        print(f"    depth limit {limit}: {c:>4} nodes visited")
    print(f"  IDDFS total visited : {i_count}   (found={i_found})")
    print(f"  BFS   total visited : {b_count}   (found={b_found})")
    print(f"  IDDFS / BFS ratio   : {i_count / b_count:.2f}x")


if __name__ == "__main__":
    tree = build_sample_tree()
    compare(tree, "A", "(root)")
    compare(tree, "C", "(depth 1)")
    compare(tree, "H", "(depth 2)")
    compare(tree, "M", "(depth 3)")
    compare(tree, "Q", "(depth 3, last leaf)")
    compare(tree, "Z", "(not in tree)")

    print("\nComplete trees, goal = last leaf (worst case)")
    print(f"{'b':>2} {'d':>2} {'nodes':>8} {'BFS':>8} {'IDDFS':>9} {'ratio':>7}")
    for b, d in [(2, 4), (2, 8), (3, 5), (3, 7), (4, 6), (10, 4)]:
        t = build_complete_tree(b, d)
        total_nodes = count_nodes_to_depth(t, d)
        goal = str(total_nodes - 1)
        _, bc = bfs(t, goal)
        _, ic, _ = iddfs(t, goal, max_depth=d)
        print(f"{b:>2} {d:>2} {total_nodes:>8} {bc:>8} {ic:>9} {ic/bc:>6.2f}x")


Goal = 'A' (root)
  IDDFS iterations:
    depth limit 0:    1 nodes visited
  IDDFS total visited : 1   (found=True)
  BFS   total visited : 1   (found=True)
  IDDFS / BFS ratio   : 1.00x

Goal = 'C' (depth 1)
  IDDFS iterations:
    depth limit 0:    1 nodes visited
    depth limit 1:    3 nodes visited
  IDDFS total visited : 4   (found=True)
  BFS   total visited : 3   (found=True)
  IDDFS / BFS ratio   : 1.33x

Goal = 'H' (depth 2)
  IDDFS iterations:
    depth limit 0:    1 nodes visited
    depth limit 1:    4 nodes visited
    depth limit 2:    7 nodes visited
  IDDFS total visited : 12   (found=True)
  BFS   total visited : 8   (found=True)
  IDDFS / BFS ratio   : 1.50x

Goal = 'M' (depth 3)
  IDDFS iterations:
    depth limit 0:    1 nodes visited
    depth limit 1:    4 nodes visited
    depth limit 2:   10 nodes visited
    depth limit 3:    8 nodes visited
  IDDFS total visited : 23   (found=True)
  BFS   total visited : 13   (found=True)
  IDDFS / BFS ratio   : 1.77x

Goa